In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Tuple

from matplotlib import pyplot as plt
from IPython.display import clear_output

from ffrnn.fftorch import FF
device = torch.device("cpu")

In [ ]:
dt = 0.001 # simulation time step (s)
delay_period = 200  # delay period (time steps)
num_steps = 2500  # total number of time steps
duration_trial = dt * num_steps  # duration of trial (s)
print("Duration of trial (s): ", duration_trial)

input_dim = 1
action_dim = 1
hidden_dim = 64

min_omega = 30
max_omega = 60
input_scaler = 10.0

pd_amp = 0.6
sh_amp = 1.4
# dopamine_change = 0.4

gpi_threshold = 1.0
gpi_tau = 0.02
ip_amp = 0.2
noise_amp = 0.5
sigmoid_gain = 3.0

In [ ]:
def cReLU(x):
    return torch.complex(torch.relu(torch.real(x)),
                         torch.relu(torch.imag(x)))

def cLeakyReLU(x):
    return torch.complex(torch.nn.LeakyReLU()(torch.real(x)),
                         torch.nn.LeakyReLU()(torch.imag(x)))

class PosLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(PosLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # We define the weight as a Parameter. 
        # Since we will use exp(), we initialize with small values 
        # so that exp(0) starts around 1.0.
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        # Apply the exponential function to force positivity
        # Positive Weight = exp(internal_weight)
        pos_weight = torch.exp(self.weight)
        
        return F.linear(x, pos_weight, self.bias)

    def extra_repr(self):
        return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'


class ComplexLinear(torch.nn.Module):
    __constants__ = ["in_features", "out_features"]
    in_features: int
    out_features: int
    weight: torch.Tensor

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        device=None,
        dtype=torch.cfloat,
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = torch.nn.Parameter(torch.empty((out_features, in_features), **factory_kwargs))
        if bias:
            self.bias = torch.nn.Parameter(torch.empty(out_features, **factory_kwargs))
        else:
            self.register_parameter("bias", None)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.linear(input, self.weight, self.bias)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"


class SingledtResHopf(nn.Module):
    def __init__(self, units, min_omega, max_omega, dt, 
                 train_omegas=True, 
                 input_scaler=5.0) -> None:
        super().__init__()

        self.units = units
        self.dt = dt
        self.input_scaler = input_scaler
        
        # Buffers for constants (avoids moving to device manually)
        self.register_buffer('min_omega', torch.tensor(min_omega * 2 * torch.pi))
        self.register_buffer('omega_range', torch.tensor((max_omega - min_omega) * 2 * torch.pi))
        self.register_buffer('mu0', torch.tensor(1.0))
        
        # Use nn.Parameter instead of autograd.Variable
        self.omegas = nn.Parameter(torch.randn(1, units), requires_grad=train_omegas)

    def forward(self, X, r, phi):
        # Pre-calculate trig values once to reuse
        cos_phi = torch.cos(phi)
        sin_phi = torch.sin(phi)
        
        omegas_scaled = torch.sigmoid(self.omegas) * self.omega_range + self.min_omega
        input_r = self.input_scaler * torch.abs(X)
        input_phi = self.input_scaler * X.real * sin_phi
        
        r_new = r + ((self.mu0 - r**2) * r + input_r) * self.dt
        phi_new = phi + (omegas_scaled - input_phi) * self.dt            

        # z = torch.complex(r_new * torch.cos(phi_new), 
        #                   r_new * torch.sin(phi_new))
        z = torch.concat([r_new * torch.cos(phi_new), r_new * torch.sin(phi_new)], dim=1)  # Shape: (batch, units, 2)
        return z, r_new, phi_new

In [ ]:
class BG(nn.Module):

    def __init__(self, input_dim, action_dim, units, d_flag=0):

        super(BG, self).__init__()

        self.input_dim = input_dim
        self.action_dim = action_dim
        self.units = units

        self.d_flag = d_flag

        self.ff = nn.Linear(input_dim, units)

        self.d1 = SingledtResHopf(units, min_omega=min_omega, max_omega=max_omega, 
                                  dt=dt, input_scaler=input_scaler)
        self.d2 = SingledtResHopf(units, min_omega=min_omega, max_omega=max_omega, 
                                  dt=dt, input_scaler=input_scaler)
        self.dp = nn.Linear(2*self.units, action_dim)
        self.ip = nn.Linear(2*self.units, action_dim)

        self.critic = nn.Linear(2*self.units, 1)

        self.gpi_tau = gpi_tau
        self.ip_amp = ip_amp
        self.noise_amp = noise_amp
        self.sigmoid_gain = sigmoid_gain

    def forward(self, state):

        v_prev = torch.zeros(1, 1, device=state.device)
        gpi = torch.zeros(1, self.action_dim, device=state.device)
        r1 = torch.ones(1, self.units, device=state.device)
        r2 = torch.ones(1, self.units, device=state.device)
        phi1 = torch.randn(1, self.units, device=state.device)/10
        phi2 = torch.randn(1, self.units, device=state.device)/10

        value_monitor = torch.zeros(state.shape[0], 1, device=state.device)
        gpi_monitor = torch.zeros(state.shape[0], self.action_dim, device=state.device)
        d1_monitor = torch.zeros(state.shape[0], 2*self.units, device=state.device)
        d2_monitor = torch.zeros(state.shape[0], 2*self.units, device=state.device)
        lD1_monitor = torch.zeros(state.shape[0], 1, device=state.device)
        lD2_monitor = torch.zeros(state.shape[0], 1, device=state.device)

        for t in range(state.shape[0]):

            x = state[t:t+1, :]
            d1out, r1, phi1 = self.d1(x, r1, phi1)
            d2out, r2, phi2 = self.d2(x, r2, phi2)

            value = self.critic(d1out)
            value_monitor[t] = value
            delV = (value - v_prev)
            v_prev = value

            # pathology condition
            if self.d_flag == 1:
                delV = pd_amp * delV

            elif self.d_flag == 2:
                delV = sh_amp * delV
                
            lD1 = torch.sigmoid(self.sigmoid_gain * delV)
            lD2 = torch.sigmoid(-self.sigmoid_gain * delV)

            lD1_monitor[t] = lD1
            lD2_monitor[t] = lD2
            d1_monitor[t] = d1out
            d2_monitor[t] = d2out

            dp_out = torch.relu(self.dp(d1out * lD1))
            ip_out = torch.relu(self.ip(d2out * lD2))
            
            noise = self.noise_amp * torch.randn_like(ip_out) * 2*lD2 - lD2

            gpi = gpi + (dp_out - self.ip_amp*(ip_out + noise)) * self.gpi_tau
            gpi_monitor[t] = gpi

        zipper = {'d1': d1_monitor, 'dp': dp_out,
                  'gpi': gpi, 'value': value_monitor,
                  'd2': d2_monitor, 'ip': ip_out,
                  'lD1': lD1_monitor, 'lD2': lD2_monitor, 'noise': noise
                  }

        return(gpi_monitor, torch.relu(value_monitor).max(), zipper)

In [ ]:
def return_stimuli(cue_length, num_steps, delay_period):

    cue_scalr = 5.0
    delay_scalr = 2.0

    cue = np.ones(cue_length) * cue_scalr
    delay = np.ones(delay_period) * delay_scalr
    padd = np.ones(num_steps - cue_length - delay_period)
    x = np.concatenate([cue, delay, padd])
    x = x * np.sin(2*np.pi*5*np.arange(num_steps)/150)  # 10 Hz modulation
    target = 2*cue_length + delay_period
    if target >= num_steps:
        raise ValueError("Target time exceeds total number of steps.")
    return np.expand_dims(x, 1), target

# x, t = return_stimuli(1000, num_steps, delay_period)
# print(x.shape)
# plt.plot(x[:,0])
# plt.axvline(t, color='red')

In [ ]:
def test_model(model, test_cue_array, num_steps, delay_period, threshold, device):
    
    pred_history = [[] for _ in range(len(test_cue_array))]
    mae_history = [[] for _ in range(len(test_cue_array))]
    num_iter = 50

    for _ in range(num_iter):
        cue = np.random.choice(test_cue_array)
        x, target_t = return_stimuli(cue, num_steps, delay_period)
        x = torch.tensor(x, dtype=torch.float32, device=device)
        out, _, _ = model(x)
        pred_t = torch.where(out >= threshold)[0]
        pred_t = pred_t[0].item() if len(pred_t)>0 else (num_steps-1)
        actual_pred_t = pred_t - cue - delay_period
        mae_current = np.abs(actual_pred_t - target_t)
        mae_history[test_cue_array.index(cue)].append(mae_current)
        pred_history[test_cue_array.index(cue)].append(actual_pred_t)

    plt.subplot(121)
    plt.errorbar(test_cue_array, 
                [np.mean(pred_history[i]) for i in range(len(test_cue_array))],
                yerr=[np.std(pred_history[i]) for i in range(len(test_cue_array))],
                fmt='o', capsize=5)
    plt.xlabel("Cue Length")
    plt.xticks(test_cue_array)
    plt.ylabel("Predicted Time")

    plt.subplot(122)
    plt.plot(test_cue_array, 
             [np.std(mae_history[i]) for i in range(len(test_cue_array))])
    plt.xlabel("Cue Length")
    plt.xticks(test_cue_array)
    plt.ylabel("Std of predicted Time")
    plt.tight_layout()
    plt.show()

In [ ]:
disease_flag = 0 # 0: healthy, 1: hypodopaminergic, 2: hyperdopaminergic

model = BG(input_dim=input_dim, action_dim=action_dim, 
           units=hidden_dim, d_flag=disease_flag)

opt = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 6_000

test_cue_array = [200, 400, 600, 800, 1000]
train_cue_array = np.arange(200, 1050, 50)

mae_monitor = []

for epoch in range(num_epochs):

    if epoch == 0:
        print("You're running the model in d_flag =", model.d_flag)
    
    cue = np.random.choice(test_cue_array)
    x, target_t = return_stimuli(cue, num_steps, delay_period)
    x = torch.tensor(x, dtype=torch.float32, device=device)

    out, value, _ = model(x)
    
    pred_t = torch.where(out >= gpi_threshold)[0]
    pred_t = pred_t[0].item() if len(pred_t)>0 else (num_steps-1)

    error = torch.abs(out[target_t] - gpi_threshold)
    reward = torch.exp(-error/1.0)

    td = reward - value

    if model.d_flag == 1:
        td = pd_amp * td
    elif model.d_flag == 2:
        td = sh_amp * td
        
    critic_loss = (td**2).reshape(error.shape)
    loss = critic_loss - value

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)   # tighter clip
    opt.step()
    
    mae_current = np.abs(pred_t - target_t)
    mae_monitor.append(mae_current)

    if (epoch+1) % 2000 == 0:

        plt.plot(mae_monitor)
        plt.xlabel("Epochs")
        plt.ylabel("MAE")
        plt.show()
        
        test_model(model, test_cue_array, num_steps, delay_period, gpi_threshold, device)
        clear_output(wait=True)

    if (epoch+1) % 1000 == 0:
        torch.save(model.state_dict(), f"trep{disease_flag}.pt")

In [ ]:
num_iter = 100

master_pred_monitor = []

for d_flag in [0, 1, 2]:

    model = BG(input_dim=input_dim, action_dim=action_dim, 
               units=hidden_dim, d_flag=d_flag)
    model.load_state_dict(torch.load(f"trep{d_flag}.pt"))

    cue_pred_monitor = []
    for cue in test_cue_array:
        pred_times = []
        for iter in range(num_iter):
            x, target_t = return_stimuli(cue, num_steps, delay_period)
            x = torch.tensor(x, dtype=torch.float32, device=device)
            out, _, _ = model(x)
            pred_t = torch.where(out >= gpi_threshold)[0]
            pred_t = pred_t[0].item() if len(pred_t)>0 else (num_steps-1)
            actual_pred_t = pred_t -cue-delay_period # Adjust for delay and cue length
            pred_times.append(actual_pred_t)
        cue_pred_monitor.append(pred_times)
    
    master_pred_monitor.append(cue_pred_monitor)
    
master_pred_monitor = np.array(master_pred_monitor)
master_pred_monitor.shape

In [ ]:
master_pred_monitor = np.load('master_pred_monitor.npy', allow_pickle=True)
test_cue_array = [200, 400, 600, 800, 1000]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# INPUT
# =========================
predictions = master_pred_monitor          # shape (3, 5, 100)
cue_times  = np.array(test_cue_array)      # length 5
conditions = ['NC', 'PD', 'SCZ']
plt.rcParams['font.size'] = 13
# =========================
# COLORS (BRIGHT & FIXED)
# =========================
cmap = plt.get_cmap('tab10')
cue_colors = [cmap(i) for i in range(len(cue_times))]

# =========================
# FIGURE (ONLY ONE ROW NOW)
# =========================
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

# =========================
# PLOTTING
# =========================
for c in range(3):

    ax = axes[c]
    mean_times = []

    for k, t in enumerate(cue_times):

        y = predictions[c, k, :]
        mean_times.append(y.mean())
        color = cue_colors[k]

        # ---- Half violin (left)
        vp = ax.violinplot(
            y,
            positions=[t],
            widths=0.15 * t,
            showmeans=False,
            showextrema=False,
            showmedians=False
        )

        for body in vp['bodies']:
            body.set_facecolor(color)
            body.set_alpha(0.45)
            body.set_clip_path(
                plt.Rectangle(
                    (t - 0.15*t, ax.get_ylim()[0]),
                    0.15*t,
                    ax.get_ylim()[1] - ax.get_ylim()[0],
                    transform=ax.transData
                )
            )

        # ---- Rain
        x = np.random.normal(t + 0.05*t, 0.015*t, size=len(y))
        ax.scatter(x, y, s=12, color=color, alpha=0.6)

        # ---- Mean bar
        ax.plot([t - 0.02*t, t + 0.02*t],
                [y.mean(), y.mean()],
                color='k', linewidth=2)

    # =========================
    # SLOPE (OLS) + DASHED FIT
    # =========================
    mean_times = np.array(mean_times)
    slope, intercept = np.polyfit(cue_times, mean_times, 1)

    # Fitted line (smooth for aesthetics)
    t_fit = np.linspace(cue_times.min(), cue_times.max(), 200)
    y_fit = intercept + slope * t_fit
    ax.plot(t_fit, y_fit, linestyle='--', linewidth=2, color='k', alpha=0.7)

    ax.legend(
        handles=[plt.Line2D([], [], linestyle='')],   
        labels = [f"Slope = {slope:.3f}"],
        frameon=False,
        loc='upper left',
    )

    ax.set_title(conditions[c], fontsize=13)
    ax.set_xlabel('Target interval (ms)')
    if c==0:
        ax.set_ylabel('Reproduced time (ms)')
    ax.set_xticks(cue_times)
    ax.set_xlim(cue_times[0]*0.55, cue_times[-1]*1.15)
    ax.set_yticks(cue_times)
    ax.set_ylim([0, cue_times[-1]*1.2])

plt.tight_layout()
plt.savefig("reproduced_vs_expected.png", dpi=400)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, ks_2samp
from itertools import combinations

# =========================
# INPUT
# =========================
predictions = master_pred_monitor          # (3, 5, 100)
cue_times  = np.array(test_cue_array)      # length 5
conditions = ['NC', 'PD', 'SCZ']

# =========================
# COLORS (SAME AS BEFORE)
# =========================
cmap = plt.get_cmap('tab10')
cue_colors = [cmap(i) for i in range(len(cue_times))]

# =========================
# FIGURE
# =========================
plt.rcParams['font.size'] = 14

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# =========================
# LOOP CONDITIONS
# =========================
for c in range(3):

    # =====================================================
    # ROW 1 — CV BAR PLOT
    # =====================================================
    ax = axes[0, c]

    means = predictions[c].mean(axis=1)
    stds  = predictions[c].std(axis=1)
    cvs   = stds / means
    cv_std = np.std(cvs)

    bar_width = 0.08 * cue_times.mean()

    ax.bar(
        cue_times,
        cvs,
        width=bar_width,
        color=cue_colors,
        edgecolor='k',
        alpha=0.9
    )

    ax.set_title(conditions[c])
    ax.set_xlabel('Target interval (ms)')
    if c== 0:
        ax.set_ylabel('CV')

    ax.legend(
    handles=[plt.Line2D([], [], linestyle='')],
    labels=[f"STD(CV) = {cv_std:.3f}"],
    frameon=False,
    loc='upper right')

    ax.set_ylim([0, 0.13])

    # =====================================================
    # ROW 2 — DISTRIBUTION COLLAPSE
    # =====================================================
    ax = axes[1, c]

    normalized = []
    for k in range(len(cue_times)):
        y = predictions[c, k, :]
        y_norm = y / y.mean()
        normalized.append(y_norm)

        # KDE (non-Gaussian assumption)
        kde = gaussian_kde(y_norm)
        x = np.linspace(0.5, 1.5, 400)
        ax.plot(x, kde(x), color=cue_colors[k], linewidth=2, alpha=0.9)

    # ---- Quantify collapse (mean pairwise KS)
    ks_vals = []
    for i, j in combinations(range(len(normalized)), 2):
        ks_vals.append(ks_2samp(normalized[i], normalized[j]).statistic)

    collapse_score = np.mean(ks_vals)

    ax.set_xlabel('Normalized reproduced time')
    if c==0:
        ax.set_ylabel('Density')
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim([0, 13.0])

    ax.legend(
        handles=[plt.Line2D([], [], linestyle='')],
        labels=[f"Mean KS = {collapse_score:.3f}"],
        frameon=False,
        loc='upper right'
    )

# =========================
# FINAL
# =========================
plt.tight_layout()
plt.savefig("scalar_collapse.png", dpi=400)

In [ ]:
np.save("master_pred_monitor.npy", master_pred_monitor)